# Prompt Strategy Test Notebook

Notebook này dùng để test output của prompt theo từng strategy: `zero-shot`, `few-shot`, `cot`.

Mục tiêu:
1. So sánh phản hồi giữa các strategy trên cùng câu.
2. Kiểm tra format tuple có đúng chuẩn hay không.
3. Kiểm tra span constraint: cụm từ dự đoán phải xuất hiện trong câu gốc.

In [ ]:
# Cell 2 · Setup imports and paths
import os
import re
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if (ROOT / 'llm_eval').exists():
    PROJECT_ROOT = ROOT
elif ROOT.name == 'llm_eval' and (ROOT / 'prompts.py').exists():
    PROJECT_ROOT = ROOT.parent
else:
    PROJECT_ROOT = ROOT

LLM_EVAL_DIR = PROJECT_ROOT / 'llm_eval'
sys.path.insert(0, str(LLM_EVAL_DIR))

from prompts import build_messages
# from client import OpenAICompatibleClient, HuggingFaceLocalClient

print('Project root:', PROJECT_ROOT)
print('llm_eval dir:', LLM_EVAL_DIR)

Project root: /home/haiyan/msc-project
llm_eval dir: /home/haiyan/msc-project/llm_eval


In [ ]:
# Cell 3 · Configuration
PROVIDER = 'openrouter'          # openrouter | hf-local
MODEL = 'openai/gpt-4o-mini'    # or HF model id
BASE_URL = 'https://openrouter.ai/api/v1'
API_KEY_ENV = 'OPENROUTER_API_KEY'

HF_DTYPE = 'auto'                # auto | float16 | bfloat16
HF_LOAD_IN_4BIT = False

TEMPERATURE = 0.0
MAX_OUTPUT_TOKENS = 256

DATASET = 'vcom-data'            # vcom-data | camera-coqe | t5-camera-coqe-data
LANGUAGE = 'vi'                  # vi | en | auto

STRATEGIES = ['zero-shot', 'few-shot', 'cot']

TEST_SENTENCES = [
    'Bên cạnh đó, iPhone 14 được nâng cấp bộ nhớ lên đến 6GB RAM cao hơn iPhone 13 đến 2GB RAM, cho khả năng đa nhiệm tốt hơn.',
    'Tương tự, thì ống kính góc rộng không có quá nhiều sự khác biệt so với ống kính chính.',
    'Bạn có thể selfie và sử dụng ở bể bơi mà không hề sợ bị hỏng máy.',
]

print('Configured provider:', PROVIDER)
print('Configured model   :', MODEL)

Configured provider: openrouter
Configured model   : openai/gpt-4o-mini


In [ ]:
# Preview prompts only (no model call)
# This helps inspect exactly what the current prompt templates look like.

sentence_preview = TEST_SENTENCES[0] if TEST_SENTENCES else ''
print('Preview sentence:', sentence_preview)

for strategy in STRATEGIES:
    messages = build_messages(
        sentence=sentence_preview,
        language=LANGUAGE,
        dataset=DATASET,
        strategy=strategy,
    )

    print('=' * 120)
    print(f'STRATEGY: {strategy}')
    print('-' * 120)
    print('[SYSTEM]')
    print(messages[0]['content'])
    print('-' * 120)
    print('[USER]')
    print(messages[1]['content'])
    print()

Preview sentence: Bên cạnh đó, iPhone 14 được nâng cấp bộ nhớ lên đến 6GB RAM cao hơn iPhone 13 đến 2GB RAM, cho khả năng đa nhiệm tốt hơn.
STRATEGY: zero-shot
------------------------------------------------------------------------------------------------------------------------
[SYSTEM]
Bạn là một mô hình trích xuất thông tin cho bài toán khai thác quan điểm so sánh. Cho một câu, hãy trích xuất tất cả bộ năm thành phần so sánh trong câu (comparative quintuples). Mỗi quintuple có 5 thành phần: [S] là chủ thể (subject), [O] là đối tượng so sánh (object), [A] là một thuộc tính được so sánh (aspect), [P] là từ/cụm từ so sánh (comparative predicate), [L] là nhãn quan hệ so sánh (comparative label). Bạn chỉ được sinh kết quả theo đúng định dạng là ([S] ... [O] ... [A] ... [P] ... [L] ...). Nếu câu có nhiều có nhiều quan hệ so sánh, hãy sinh các quintuple được ngăn cách bằng ' ; '. Mọi cụm từ được trích xuất cho [S], [O], [A], [P] phải là nguyên văn xuất hiện trong câu gốc. Nếu đối tượng so

In [ ]:
# Cell 4 · Build client
if PROVIDER == 'openrouter':
    client = OpenAICompatibleClient(
        model=MODEL,
        base_url=BASE_URL,
        api_key_env=API_KEY_ENV,
        temperature=TEMPERATURE,
        max_output_tokens=MAX_OUTPUT_TOKENS,
    )
elif PROVIDER == 'hf-local':
    client = HuggingFaceLocalClient(
        model=MODEL,
        temperature=TEMPERATURE,
        max_output_tokens=MAX_OUTPUT_TOKENS,
        dtype=HF_DTYPE,
        load_in_4bit=HF_LOAD_IN_4BIT,
    )
else:
    raise ValueError(f'Unsupported PROVIDER: {PROVIDER}')

print('Client initialized.')

In [ ]:
# Cell 5 · Format validators
TUPLE_RE = re.compile(
    r'\[S\]\s*(.*?)\s*\[O\]\s*(.*?)\s*\[A\]\s*(.*?)\s*\[P\]\s*(.*?)\s*\[L\]\s*(.*?)(?=\)|\n|;|$)',
    re.DOTALL,
)

EMPTY_TUPLE = '([S] [UNK] [O] [UNK] [A] [UNK] [P] [UNK] [L] [UNK])'

def parse_tuples(output_text: str):
    tuples = []
    for part in (output_text or '').split(';'):
        m = TUPLE_RE.search(part.strip().strip('()'))
        if m:
            s, o, a, p, l = [x.strip() for x in m.groups()]
            tuples.append({'S': s, 'O': o, 'A': a, 'P': p, 'L': l})
    return tuples

def validate_format(output_text: str):
    text = (output_text or '').strip()
    tuples = parse_tuples(text)
    ok_structure = (text == EMPTY_TUPLE) or (len(tuples) > 0)
    return {
        'ok_structure': ok_structure,
        'tuple_count': len(tuples),
        'parsed_tuples': tuples,
    }

def validate_span_constraint(sentence: str, parsed_tuples):
    sent_low = sentence.lower()
    violations = []
    for i, t in enumerate(parsed_tuples, start=1):
        for slot in ('S', 'O', 'A', 'P'):
            val = (t.get(slot) or '').strip()
            if not val or val == '[UNK]':
                continue
            if val.lower() not in sent_low:
                violations.append({'tuple_idx': i, 'slot': slot, 'value': val})
    return {
        'ok_span_constraint': len(violations) == 0,
        'violations': violations,
    }

In [ ]:
# Cell 6 · Run prompt tests by strategy
import json
from datetime import datetime

results = []

# Raw log file: one JSON object per (sentence, strategy)
out_dir = LLM_EVAL_DIR / 'results' / 'prompt_test'
out_dir.mkdir(parents=True, exist_ok=True)
run_ts = datetime.now().strftime('%Y%m%d_%H%M%S')
raw_log_file = out_dir / f'raw_outputs__{PROVIDER}__{run_ts}.jsonl'

with open(raw_log_file, 'w', encoding='utf-8') as log_fp:
    for sent_idx, sentence in enumerate(TEST_SENTENCES, start=1):
        print('= ' * 50)
        print(f'Sentence {sent_idx}: {sentence}')

        for strategy in STRATEGIES:
            messages = build_messages(
                sentence=sentence,
                language=LANGUAGE,
                dataset=DATASET,
                strategy=strategy,
            )

            output = client.generate(messages)
            fmt = validate_format(output)
            span = validate_span_constraint(sentence, fmt['parsed_tuples'])

            record = {
                'sentence_idx': sent_idx,
                'strategy': strategy,
                'output': output,
                'ok_structure': fmt['ok_structure'],
                'tuple_count': fmt['tuple_count'],
                'ok_span_constraint': span['ok_span_constraint'],
                'violations': span['violations'],
            }
            results.append(record)

            # Persist raw trace for debugging and reproducibility.
            raw_row = {
                'timestamp': run_ts,
                'provider': PROVIDER,
                'model': MODEL,
                'dataset': DATASET,
                'language': LANGUAGE,
                'sentence_idx': sent_idx,
                'sentence': sentence,
                'strategy': strategy,
                'messages': messages,
                'output': output,
                'validation': {
                    'ok_structure': fmt['ok_structure'],
                    'ok_span_constraint': span['ok_span_constraint'],
                    'tuple_count': fmt['tuple_count'],
                    'violations': span['violations'],
                },
            }
            log_fp.write(json.dumps(raw_row, ensure_ascii=False) + '\n')

            status = 'PASS' if (fmt['ok_structure'] and span['ok_span_constraint']) else 'FAIL'
            print(f'[{strategy}] => {status}')
            print('Output:', output)
            if not span['ok_span_constraint']:
                print('Span violations:', span['violations'])
            print('-' * 100)

print('Raw output log saved to:', raw_log_file)

In [ ]:
# Cell 7 · Summary table
try:
    import pandas as pd
    df = pd.DataFrame(results)
    show_cols = ['sentence_idx', 'strategy', 'ok_structure', 'ok_span_constraint', 'tuple_count']
    print(df[show_cols].to_string(index=False))
except ImportError:
    for r in results:
        print({k: r[k] for k in ['sentence_idx', 'strategy', 'ok_structure', 'ok_span_constraint', 'tuple_count']})

In [ ]:
# Cell 8 · Save test report (optional)
import json
from datetime import datetime

out_dir = LLM_EVAL_DIR / 'results' / 'prompt_test'
out_dir.mkdir(parents=True, exist_ok=True)

ts = datetime.now().strftime('%Y%m%d_%H%M%S')
out_file = out_dir / f'prompt_test__{PROVIDER}__{ts}.json'

payload = {
    'provider': PROVIDER,
    'model': MODEL,
    'dataset': DATASET,
    'language': LANGUAGE,
    'strategies': STRATEGIES,
    'results': results,
}

with open(out_file, 'w', encoding='utf-8') as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

print('Saved report to:', out_file)